# Live Lab 2 · 3D Reconstruction
**COLMAP vs Geometric Foundation Models**

Run the cells from top to bottom. No code to write, about 10 minutes.

*Before you start: Runtime → Change runtime type → **T4 GPU**.*

| | |
|---|---|
| **Session 1** | Your SfM vs COLMAP, on the fountain |
| **Session 2** | Depth Anything 3, on the same photos |
| **Session 3** | COLMAP vs Depth Anything 3, on **our** photos |

In [ ]:
#@title Setup (1-2 min)
!pip -q install pycolmap plotly
# Depth Anything 3 is installed WITHOUT its pinned dependencies, to keep Colab's numpy 2.
# --no-warn-conflicts hides the red "dependency conflict" messages: they are harmless here.
!pip -q install --no-deps --no-warn-conflicts git+https://github.com/ByteDance-Seed/Depth-Anything-3.git
!pip -q install --no-warn-conflicts addict omegaconf einops safetensors trimesh plyfile pillow_heif imageio proglog
!pip -q install --no-deps --no-warn-conflicts moviepy==1.0.3
!wget -q https://stereo-vision.s3.eu-west-3.amazonaws.com/fountain.zip
!unzip -qo fountain.zip
print('Setup done')

In [ ]:
#@title Tools 1/4 · Load the fountain photos (just run)
import glob, os, shutil, time
from dataclasses import dataclass
import cv2
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import pandas as pd


def photo_number(path):
    """'fountain/0010.png' -> 10, so that photos are sorted 1, 2, ... 10, 11."""
    name = os.path.basename(path).split('.')[0]
    digits = ''.join(character for character in name if character.isdigit())
    return int(digits) if digits else 0


def read_true_camera_position(camera_file):
    """The dataset gives the true camera position (in meters) on line 8 of each .camera file."""
    lines = open(camera_file).read().split('\n')
    return [float(value) for value in lines[7].split()]


photo_paths = sorted(glob.glob('fountain/*.png'), key=photo_number)
photos = [cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB) for path in photo_paths]
number_of_photos = len(photos)

# Camera calibration of the fountain photos: focal lengths and image center, in pixels.
camera_matrix = np.array([[2759.48,       0, 1520.69],
                          [      0, 2764.16, 1006.81],
                          [      0,       0,       1]])

true_positions = np.array([read_true_camera_position(path + '.camera') for path in photo_paths])
print(number_of_photos, 'photos loaded')

In [ ]:
#@title Tools 2/4 · Your SfM from Lab 1, packaged (just run)
# Step 1: find interest points (SIFT) in every photo.
sift = cv2.SIFT_create()
keypoints = []     # keypoints[photo] = pixel positions (x, y) of its SIFT points
descriptors = []   # descriptors[photo] = the "fingerprint" of each SIFT point
for photo in photos:
    sift_points, sift_fingerprints = sift.detectAndCompute(photo, None)
    keypoints.append(np.array([point.pt for point in sift_points]))
    descriptors.append(sift_fingerprints)


def match_photos(photo_a, photo_b):
    """Returns a list of [keypoint in photo_a, keypoint in photo_b] that look the same."""
    matcher = cv2.BFMatcher(crossCheck=True)
    matches = matcher.match(descriptors[photo_a], descriptors[photo_b])
    return np.array([[match.queryIdx, match.trainIdx] for match in matches])


def pixels_to_normalized(pixels):
    """Pixels -> camera coordinates (removes the focal length and the image center)."""
    pixels = pixels.reshape(-1, 1, 2)
    return cv2.undistortPoints(pixels, camera_matrix, None).reshape(-1, 2)


def triangulate(pose_a, pose_b, pixels_a, pixels_b):
    """Two views of the same points -> their 3D positions. A pose is a 3x4 matrix [R | t]."""
    rays_a = pixels_to_normalized(pixels_a)
    rays_b = pixels_to_normalized(pixels_b)
    points_homogeneous = cv2.triangulatePoints(pose_a, pose_b, rays_a.T, rays_b.T)
    return (points_homogeneous[:3] / points_homogeneous[3]).T


@dataclass
class TwoViewResult:
    matches: np.ndarray       # [keypoint in photo A, keypoint in photo B] for each kept point
    rotation: np.ndarray      # rotation of camera B, relative to camera A
    translation: np.ndarray   # translation of camera B (its length is unknown: set to 1)
    points_3d: np.ndarray     # the triangulated points, in the frame of camera A
    colors: np.ndarray        # the color of each point


def two_view_sfm(photo_a, photo_b):
    """Your SfM: match -> essential matrix -> R and t -> triangulate."""
    matches = match_photos(photo_a, photo_b)
    pixels_a = keypoints[photo_a][matches[:, 0]]
    pixels_b = keypoints[photo_b][matches[:, 1]]

    # Essential matrix with RANSAC: it ignores the wrong matches.
    essential_matrix, inliers = cv2.findEssentialMat(
        pixels_a, pixels_b, camera_matrix, cv2.RANSAC, 0.999, 1.0)

    # Relative pose. The returned mask also removes points that end up behind a camera.
    _, rotation, translation, kept = cv2.recoverPose(
        essential_matrix, pixels_a, pixels_b, camera_matrix, mask=inliers.copy())
    kept = kept.ravel() > 0
    matches, pixels_a, pixels_b = matches[kept], pixels_a[kept], pixels_b[kept]

    pose_a = np.hstack([np.eye(3), np.zeros((3, 1))])
    pose_b = np.hstack([rotation, translation])
    points_3d = triangulate(pose_a, pose_b, pixels_a, pixels_b)

    columns = pixels_a[:, 0].astype(int)
    rows = pixels_a[:, 1].astype(int)
    colors = photos[photo_a][rows, columns]
    return TwoViewResult(matches, rotation, translation, points_3d, colors)

In [ ]:
#@title Tools 3/4 · Plots and alignment (just run)
def make_aligner(estimated_positions, reference_positions):
    """Finds the scale, rotation and translation that best move one camera path onto another.
    Needed because an SfM result has an arbitrary scale, orientation and origin."""
    estimated_center = estimated_positions.mean(axis=0)
    reference_center = reference_positions.mean(axis=0)
    estimated = estimated_positions - estimated_center
    reference = reference_positions - reference_center

    U, singular_values, Vt = np.linalg.svd(reference.T @ estimated)
    mirror_fix = np.eye(3)
    if np.linalg.det(U @ Vt) < 0:
        mirror_fix[2, 2] = -1
    rotation = U @ mirror_fix @ Vt
    scale = np.trace(np.diag(singular_values) @ mirror_fix) / (estimated ** 2).sum()

    def align(points):
        points = np.asarray(points)
        return scale * (points - estimated_center) @ rotation.T + reference_center
    return align


def camera_positions_from_poses(rotations, translations):
    """A camera pose maps world -> camera. The camera position in the world is -R^T t."""
    return np.array([(-rotation.T @ translation).ravel()
                     for rotation, translation in zip(rotations, translations)])


# A 3D view with too many points becomes too heavy for the browser: we keep 25 000 at most.
MAX_POINTS_IN_3D_VIEW = 25000

COLORS = ['#e6194b', '#3cb44b', '#4363d8', '#f58231', '#911eb4',
          '#42d4f4', '#f032e6', '#bfef45', '#469990', '#9A6324']


def show_3d(point_clouds, camera_positions=None, title='', camera_rotations=None):
    """point_clouds = list of (points, colors, name). Opens an interactive 3D view.
    camera_rotations (world -> camera) are used to turn the view upright: in a camera frame,
    the y axis points DOWN, but the 3D viewer expects z to point up."""
    figure = go.Figure()
    for points, colors, name in point_clouds:
        points = np.asarray(points)
        is_valid = np.isfinite(points).all(axis=1)
        # Hide the few far-away points that would squash the view.
        low = np.percentile(points[is_valid], 2, axis=0)
        high = np.percentile(points[is_valid], 98, axis=0)
        margin = (high - low) * 0.3
        is_close = ((points > low - margin) & (points < high + margin)).all(axis=1)
        selected = np.where(is_valid & is_close)[0]
        selected = np.random.default_rng(0).permutation(selected)[:MAX_POINTS_IN_3D_VIEW]
        if isinstance(colors, str):
            marker_colors = colors
        else:
            marker_colors = ['rgb(%d,%d,%d)' % tuple(colors[i]) for i in selected]
        figure.add_trace(go.Scatter3d(
            x=points[selected, 0].round(2), y=points[selected, 1].round(2),
            z=points[selected, 2].round(2),
            mode='markers', name=name, marker=dict(size=1.5, color=marker_colors)))
    if camera_positions is not None:
        figure.add_trace(go.Scatter3d(
            x=camera_positions[:, 0], y=camera_positions[:, 1], z=camera_positions[:, 2],
            mode='markers+lines', name='cameras',
            marker=dict(size=5, color='red'), line=dict(color='red')))
    hidden_axis = dict(visible=False)
    figure.update_layout(title=title, height=600, margin=dict(l=0, r=0, t=40, b=0),
                         scene=dict(aspectmode='data', xaxis=hidden_axis,
                                    yaxis=hidden_axis, zaxis=hidden_axis))
    if camera_rotations is not None:
        rotations = np.asarray(camera_rotations)
        down = rotations[:, 1, :].mean(axis=0)       # the y axis of the cameras, in the 3D frame
        forward = rotations[:, 2, :].mean(axis=0)    # the z axis of the cameras: where they look
        up = -down / np.linalg.norm(down)
        eye = -1.7 * forward / np.linalg.norm(forward) + 0.7 * up   # look from behind the cameras
        figure.update_layout(scene_camera=dict(
            up=dict(x=up[0], y=up[1], z=up[2]), eye=dict(x=eye[0], y=eye[1], z=eye[2])))
    figure.show()


def plot_camera_paths(paths, title):
    """paths = list of (camera positions, label, color).
    Left: the camera paths seen from above. Right: error of each camera, in cm."""
    figure, (map_view, error_view) = plt.subplots(1, 2, figsize=(14, 5))
    map_view.plot(true_positions[:, 0], true_positions[:, 1], 'k-o', linewidth=3,
                  markersize=7, label='true positions')
    bar_width = 0.8 / len(paths)
    for index, (positions, label, color) in enumerate(paths):
        aligned = make_aligner(positions, true_positions)(positions)
        error_cm = 100 * np.linalg.norm(aligned - true_positions, axis=1)
        map_view.plot(aligned[:, 0], aligned[:, 1], '-o', color=color, linewidth=2, label=label)
        bar_shift = (index - (len(paths) - 1) / 2) * bar_width
        error_view.bar(np.arange(1, len(positions) + 1) + bar_shift, error_cm, bar_width,
                       color=color, label=f'{label}: mean {error_cm.mean():.0f} cm')
    map_view.set_aspect('equal', 'datalim')
    map_view.set_title(title)
    map_view.set_xlabel('x (m)')
    map_view.set_ylabel('y (m)')
    map_view.legend()
    error_view.set_title('Error of each camera (cm). Lower is better')
    error_view.set_xlabel('photo')
    error_view.legend()
    plt.tight_layout()
    plt.show()

# **Session 1 · Your SfM vs COLMAP**

## **1.1 · Your SfM on 2 photos**
In Lab 1 you built: match, essential matrix, R and t, triangulate.
It is packaged in one function, `two_view_sfm`. It works well on one pair of photos.

In [ ]:
#@title Run your SfM on photos 1 and 2
result = two_view_sfm(0, 1)
print(len(result.points_3d), '3D points')
show_3d([(result.points_3d, result.colors, 'photos 1-2')], title='Your SfM, 2 photos',
        camera_rotations=[np.eye(3)])

## **1.2 · Your SfM on 11 photos**
Same function, one pair after the other. We stack all the results.

In [ ]:
#@title Run every pair and stack the results
two_view_results = []
for photo in range(number_of_photos - 1):
    result = two_view_sfm(photo, photo + 1)
    two_view_results.append(result)

stacked_points = np.vstack([result.points_3d for result in two_view_results])
clouds = []
for pair_index, result in enumerate(two_view_results):
    name = f'photos {pair_index + 1}-{pair_index + 2}'
    clouds.append((result.points_3d, COLORS[pair_index % len(COLORS)], name))
show_3d(clouds, title=f'{len(stacked_points)} points stacked. One color = one pair of photos',
        camera_rotations=[np.eye(3)])

It breaks. Three problems:
1. **Different scale**: every pair assumes the two cameras are 1 unit apart.
2. **Duplicates**: the same corner is triangulated again in every pair.
3. **Drift**: chaining pair after pair adds up small errors.

In [ ]:
#@title The 3 problems, measured
figure, (scale_plot, duplicate_plot) = plt.subplots(1, 2, figsize=(13, 4))

# Problem 1: the distance between consecutive cameras.
true_distances = np.linalg.norm(np.diff(true_positions, axis=0), axis=1)
pair_numbers = np.arange(1, number_of_photos)
scale_plot.bar(pair_numbers - 0.2, true_distances, 0.4, color='black', label='true distance (m)')
scale_plot.bar(pair_numbers + 0.2, np.ones(number_of_photos - 1), 0.4, color='#e6194b',
               label='what your SfM assumes')
scale_plot.set_title('1. Different scale: distance between two cameras')
scale_plot.set_xlabel('pair of photos')
scale_plot.legend()

# Problem 2: a keypoint used by pair (k-1, k) is used again by pair (k, k+1).
duplicates = []
for pair_index in range(number_of_photos - 2):
    keypoints_of_first_pair = set(two_view_results[pair_index].matches[:, 1])
    keypoints_of_next_pair = set(two_view_results[pair_index + 1].matches[:, 0])
    duplicates.append(len(keypoints_of_first_pair & keypoints_of_next_pair))
duplicate_plot.bar(np.arange(2, number_of_photos), duplicates, color='#f58231')
duplicate_plot.set_title(f'2. Duplicates: {sum(duplicates)} points built twice')
duplicate_plot.set_xlabel('photo')
duplicate_plot.set_ylabel('points')
plt.tight_layout()
plt.show()

# Problem 3: chain the relative poses, pair after pair.
chained_rotations = [np.eye(3)]
chained_translations = [np.zeros((3, 1))]
for result in two_view_results:
    previous_rotation = chained_rotations[-1]
    previous_translation = chained_translations[-1]
    chained_rotations.append(result.rotation @ previous_rotation)
    chained_translations.append(result.rotation @ previous_translation + result.translation)
chained_positions = camera_positions_from_poses(chained_rotations, chained_translations)

plot_camera_paths([(chained_positions, 'your SfM, chained', '#e6194b')],
                  '3. Drift: camera path seen from above')

## **1.3 · What COLMAP adds: 3 bricks**
| Brick | What it does | What it fixes |
|---|---|---|
| **1 · Tracks** | follows each point through all the photos | the duplicates |
| **2 · PnP** | places each new camera using the 3D points already built | the scale |
| **3 · Bundle adjustment** | moves all cameras and all points together, to fit the photos | the drift |

We add them one by one and look at the result before and after.
Brick 2 needs the tracks (they tell which 3D points a new photo can see), so tracks come first.

In [ ]:
#@title Tools 4/4 · The 3 bricks (just run, no need to read)
from scipy.optimize import least_squares
from scipy.sparse import coo_matrix


# ---------- Brick 1 · Tracks ----------
def build_tracks(two_view_results):
    """A track follows ONE real point through several photos: {photo: keypoint in that photo}."""
    tracks = []
    track_of_keypoint = {}   # (photo, keypoint) -> index of the track it belongs to
    for photo_a, result in enumerate(two_view_results):
        photo_b = photo_a + 1
        for keypoint_a, keypoint_b in result.matches:
            track_index = track_of_keypoint.get((photo_a, keypoint_a))
            if track_index is None:
                # First time we meet this point: start a new track.
                track_index = len(tracks)
                tracks.append({photo_a: keypoint_a})
                track_of_keypoint[(photo_a, keypoint_a)] = track_index
            tracks[track_index][photo_b] = keypoint_b
            track_of_keypoint[(photo_b, keypoint_b)] = track_index
    return tracks


def plot_track_statistics(tracks, number_of_stacked_points):
    figure, (count_plot, length_plot) = plt.subplots(1, 2, figsize=(13, 4))

    labels = ['before\n(stacked pairs)', 'after\n(tracks)']
    counts = [number_of_stacked_points, len(tracks)]
    bars = count_plot.bar(labels, counts, color=['#e6194b', '#2EAD6B'])
    count_plot.bar_label(bars)
    count_plot.set_title('Number of 3D points: one point is now built once')

    lengths = [len(track) for track in tracks]
    photo_counts = np.arange(2, number_of_photos + 1)
    points_per_length = [lengths.count(length) for length in photo_counts]
    bars = length_plot.bar(photo_counts, points_per_length, color='#2EAD6B')
    length_plot.bar_label(bars)
    length_plot.set_xticks(photo_counts)
    length_plot.set_xlabel('number of photos where the point is seen')
    length_plot.set_ylabel('number of points')
    length_plot.set_title('Each point, seen in how many photos?')
    plt.tight_layout()
    plt.show()


def show_one_track(tracks):
    """Shows the longest track: the same real point, cropped in each photo."""
    track = max(tracks, key=len)
    photos_to_show = sorted(track)[:6]
    figure, axes = plt.subplots(1, len(photos_to_show), figsize=(3 * len(photos_to_show), 3))
    for axis, photo in zip(axes, photos_to_show):
        x, y = keypoints[photo][track[photo]].astype(int)
        left, top = max(0, x - 150), max(0, y - 150)
        axis.imshow(photos[photo][top:y + 150, left:x + 150])
        axis.plot(x - left, y - top, 'o', markersize=18, markerfacecolor='none',
                  markeredgecolor='cyan', markeredgewidth=3)
        axis.set_title(f'photo {photo + 1}')
        axis.axis('off')
    plt.suptitle(f'One track: the same point, found in {len(track)} photos')
    plt.show()


# ---------- Brick 2 · PnP ----------
def build_new_points(tracks, points_3d, rotations, translations, photo):
    """Triangulates the points seen in `photo` and in the photo before, not built yet."""
    previous = photo - 1
    new_tracks = [index for index, track in enumerate(tracks)
                  if previous in track and photo in track and np.isnan(points_3d[index, 0])]
    if len(new_tracks) == 0:
        return
    pixels_previous = np.array([keypoints[previous][tracks[i][previous]] for i in new_tracks])
    pixels_photo = np.array([keypoints[photo][tracks[i][photo]] for i in new_tracks])
    pose_previous = np.hstack([rotations[previous], translations[previous]])
    pose_photo = np.hstack([rotations[photo], translations[photo]])
    new_points = triangulate(pose_previous, pose_photo, pixels_previous, pixels_photo)

    # Keep only the points that are in front of both cameras.
    depth_previous = (rotations[previous] @ new_points.T + translations[previous])[2]
    depth_photo = (rotations[photo] @ new_points.T + translations[photo])[2]
    in_front = (depth_previous > 0) & (depth_photo > 0)
    points_3d[np.array(new_tracks)[in_front]] = new_points[in_front]


def place_camera_with_pnp(tracks, points_3d, photo):
    """PnP: we know where some 3D points are, and where they appear in this photo.
    The camera pose is the one that makes the 3D points land on their pixels."""
    known = [index for index, track in enumerate(tracks)
             if photo in track and not np.isnan(points_3d[index, 0])]
    points = points_3d[known]
    pixels = np.array([keypoints[photo][tracks[index][photo]] for index in known])
    success, rotation_vector, translation, _ = cv2.solvePnPRansac(
        points, pixels, camera_matrix, None, reprojectionError=4, iterationsCount=2000)
    if not success:
        raise RuntimeError(f'PnP failed on photo {photo + 1}: not enough known 3D points')
    return cv2.Rodrigues(rotation_vector)[0], translation.reshape(3, 1)


def register_cameras(tracks, two_view_results):
    """Photo 1 is the origin, photo 2 comes from the first pair.
    Each next photo is placed with PnP, then adds its new 3D points."""
    rotations = [np.eye(3), two_view_results[0].rotation]
    translations = [np.zeros((3, 1)), two_view_results[0].translation]
    points_3d = np.full((len(tracks), 3), np.nan)   # NaN = not built yet
    build_new_points(tracks, points_3d, rotations, translations, photo=1)
    for photo in range(2, number_of_photos):
        rotation, translation = place_camera_with_pnp(tracks, points_3d, photo)
        rotations.append(rotation)
        translations.append(translation)
        build_new_points(tracks, points_3d, rotations, translations, photo)
    return rotations, translations, points_3d


# ---------- Brick 3 · Bundle adjustment ----------
def project_points(points, rotation_vector, translation):
    """Where do 3D points appear in a photo? (pixels)"""
    projected, _ = cv2.projectPoints(points, rotation_vector, translation, camera_matrix, None)
    return projected.reshape(-1, 2)


def list_observations(tracks, points_3d):
    """Every time a built 3D point is seen in a photo: which photo, which point, which pixel."""
    built_ids = np.where(~np.isnan(points_3d[:, 0]))[0]
    observed_photo, observed_point, observed_pixels = [], [], []
    for point_number, track_index in enumerate(built_ids):
        for photo, keypoint in tracks[track_index].items():
            observed_photo.append(photo)
            observed_point.append(point_number)
            observed_pixels.append(keypoints[photo][keypoint])
    return (built_ids, np.array(observed_photo), np.array(observed_point),
            np.array(observed_pixels))


def reprojection_errors(tracks, rotations, translations, points_3d):
    """Distance in pixels between where a 3D point is projected and where it was seen."""
    built_ids, observed_photo, observed_point, observed_pixels = list_observations(tracks, points_3d)
    errors = np.zeros(len(observed_photo))
    for photo in range(number_of_photos):
        on_this_photo = observed_photo == photo
        rotation_vector = cv2.Rodrigues(rotations[photo])[0]
        projected = project_points(points_3d[built_ids][observed_point[on_this_photo]],
                                   rotation_vector, translations[photo])
        errors[on_this_photo] = np.linalg.norm(projected - observed_pixels[on_this_photo], axis=1)
    return errors


def bundle_adjust(tracks, rotations, translations, points_3d):
    """Moves every camera (except photo 1) and every 3D point to reduce the reprojection error."""
    built_ids, observed_photo, observed_point, observed_pixels = list_observations(tracks, points_3d)
    free_cameras = number_of_photos - 1

    # The unknowns: 6 numbers per camera (3 rotation + 3 translation), then 3 per 3D point.
    camera_numbers = []
    for photo in range(1, number_of_photos):
        rotation_vector = cv2.Rodrigues(rotations[photo])[0].ravel()
        camera_numbers.append(np.concatenate([rotation_vector, translations[photo].ravel()]))
    start = np.concatenate([np.ravel(camera_numbers), points_3d[built_ids].ravel()])

    def residuals(unknowns):
        cameras = unknowns[:6 * free_cameras].reshape(-1, 6)
        points = unknowns[6 * free_cameras:].reshape(-1, 3)
        differences = np.zeros((len(observed_photo), 2))
        for photo in range(number_of_photos):
            on_this_photo = observed_photo == photo
            if photo == 0:
                rotation_vector, translation = np.zeros(3), np.zeros(3)   # photo 1 stays fixed
            else:
                rotation_vector, translation = cameras[photo - 1, :3], cameras[photo - 1, 3:]
            projected = project_points(points[observed_point[on_this_photo]],
                                       rotation_vector, translation)
            differences[on_this_photo] = projected - observed_pixels[on_this_photo]
        return differences.ravel()

    # Each measurement depends on ONE camera and ONE point only. Telling the solver makes it fast.
    rows, columns = [], []
    for measurement in range(len(observed_photo)):
        for coordinate in range(2):
            row = 2 * measurement + coordinate
            for number in range(3):
                rows.append(row)
                columns.append(6 * free_cameras + 3 * observed_point[measurement] + number)
            if observed_photo[measurement] > 0:
                for number in range(6):
                    rows.append(row)
                    columns.append(6 * (observed_photo[measurement] - 1) + number)
    shape = (2 * len(observed_photo), len(start))
    sparsity = coo_matrix((np.ones(len(rows), dtype=int), (rows, columns)), shape=shape)

    solution = least_squares(residuals, start, jac_sparsity=sparsity, loss='huber',
                             f_scale=1.0, x_scale='jac', max_nfev=50)

    cameras = solution.x[:6 * free_cameras].reshape(-1, 6)
    new_rotations, new_translations = [rotations[0]], [translations[0]]
    for camera in cameras:
        new_rotations.append(cv2.Rodrigues(camera[:3])[0])
        new_translations.append(camera[3:].reshape(3, 1))
    new_points_3d = np.full_like(points_3d, np.nan)
    new_points_3d[built_ids] = solution.x[6 * free_cameras:].reshape(-1, 3)

    # Remove the points that still do not fit one of their photos (wrong matches).
    errors = reprojection_errors(tracks, new_rotations, new_translations, new_points_3d)
    wrong_points = np.unique(observed_point[errors > 4])
    new_points_3d[built_ids[wrong_points]] = np.nan
    return new_rotations, new_translations, new_points_3d


def point_colors(tracks, points_3d):
    """Color of each built point, read in the first photo that sees it."""
    colors = np.zeros((len(tracks), 3), dtype=int)
    for index, track in enumerate(tracks):
        photo = min(track)
        x, y = keypoints[photo][track[photo]].astype(int)
        colors[index] = photos[photo][y, x]
    return colors

### **Brick 1 · Tracks**
A track follows one real point through all the photos.
Before: the same corner is built again in every pair. After: it is built once.

In [ ]:
#@title Brick 1 · Tracks: link the same point across photos
tracks = build_tracks(two_view_results)
plot_track_statistics(tracks, number_of_stacked_points=len(stacked_points))
show_one_track(tracks)

### **Brick 2 · PnP**
Each new camera is placed using the 3D points we already have, so all cameras share one scale.

In [ ]:
#@title Brick 2 · PnP: place each new camera on the existing 3D points
rotations_pnp, translations_pnp, points_3d_pnp = register_cameras(tracks, two_view_results)
positions_pnp = camera_positions_from_poses(rotations_pnp, translations_pnp)

plot_camera_paths([(chained_positions, 'before: chained pairs', '#e6194b'),
                   (positions_pnp, 'after: PnP', '#4363d8')],
                  'PnP: camera path seen from above')

### **Brick 3 · Bundle adjustment**
All cameras and all points are moved together, to match the photos as well as possible.
We measure it with the **reprojection error**: the distance, in pixels,
between where a 3D point is projected and where it was really seen. Lower is better.

In [ ]:
#@title Brick 3 · Bundle adjustment: refine everything together (about 10 s)
start_time = time.time()
rotations_ba, translations_ba, points_3d_ba = bundle_adjust(
    tracks, rotations_pnp, translations_pnp, points_3d_pnp)
positions_ba = camera_positions_from_poses(rotations_ba, translations_ba)
print(f'Done in {time.time() - start_time:.0f} s')

errors_before = reprojection_errors(tracks, rotations_pnp, translations_pnp, points_3d_pnp)
errors_after = reprojection_errors(tracks, rotations_ba, translations_ba, points_3d_ba)
def describe(errors):
    share_above_1px = 100 * np.mean(errors > 1)
    return f'mean {errors.mean():.2f} px, {share_above_1px:.0f}% of points above 1 px'

bins = np.linspace(0, 3, 31)
plt.figure(figsize=(8, 4))
plt.hist(np.clip(errors_before, 0, 3), bins, alpha=0.6, color='#e6194b',
         label='before: ' + describe(errors_before))
plt.hist(np.clip(errors_after, 0, 3), bins, alpha=0.6, color='#2EAD6B',
         label='after: ' + describe(errors_after))
plt.xlabel('reprojection error (pixels). Errors above 3 px are drawn at 3')
plt.ylabel('number of measurements')
plt.title('Bundle adjustment: reprojection error')
plt.legend()
plt.show()

plot_camera_paths([(chained_positions, 'chained pairs', '#e6194b'),
                   (positions_pnp, '+ tracks + PnP', '#4363d8'),
                   (positions_ba, '+ bundle adjustment', '#2EAD6B')],
                  'Before / after, camera path seen from above')

In [ ]:
#@title Your SfM with the 3 bricks, in 3D
is_built = ~np.isnan(points_3d_ba[:, 0])
align_to_true = make_aligner(positions_ba, true_positions)
show_3d([(align_to_true(points_3d_ba[is_built]), point_colors(tracks, points_3d_ba)[is_built],
          'your SfM + 3 bricks')],
        camera_positions=align_to_true(positions_ba),
        title=f'{is_built.sum()} points, {number_of_photos} cameras, in meters')

## **1.4 · The real COLMAP**
Same 3 bricks, industrial version: features, matching of all pairs,
then incremental mapping (PnP, tracks and bundle adjustment inside).

In [ ]:
#@title Run COLMAP on the fountain (about 2 min)
import pycolmap


def run_colmap(photo_folder, output_folder):
    """Features -> matching of all pairs -> incremental mapping. Returns the biggest result."""
    shutil.rmtree(output_folder, ignore_errors=True)
    os.makedirs(output_folder)
    database = f'{output_folder}/database.db'
    pycolmap.extract_features(database, photo_folder)
    pycolmap.match_exhaustive(database)
    reconstructions = pycolmap.incremental_mapping(database, photo_folder, output_folder)
    if len(reconstructions) == 0:
        return None
    return max(reconstructions.values(), key=lambda found: found.num_reg_images())


def colmap_camera_positions(reconstruction):
    """Camera positions of the photos COLMAP managed to place, sorted by photo number."""
    images = sorted(reconstruction.images.values(), key=lambda image: photo_number(image.name))
    names = [image.name for image in images]
    positions = np.array([image.projection_center() for image in images])
    return names, positions


def colmap_camera_rotations(reconstruction):
    """Rotation (world -> camera) of each placed photo, in the same order as the positions."""
    images = sorted(reconstruction.images.values(), key=lambda image: photo_number(image.name))
    return [image.cam_from_world().matrix()[:3, :3] for image in images]


def colmap_points(reconstruction):
    points = np.array([point.xyz for point in reconstruction.points3D.values()])
    colors = np.array([point.color for point in reconstruction.points3D.values()])
    return points, colors


# COLMAP reads a folder: we copy the photos alone (without the .camera files).
os.makedirs('fountain_images', exist_ok=True)
for path in photo_paths:
    shutil.copy(path, 'fountain_images/' + os.path.basename(path))

start_time = time.time()
colmap_result = run_colmap('fountain_images', 'colmap_fountain')
colmap_names, colmap_positions = colmap_camera_positions(colmap_result)
colmap_cloud, colmap_colors = colmap_points(colmap_result)
print(f'{colmap_result.num_reg_images()} / {number_of_photos} photos placed, '
      f'{colmap_result.num_points3D()} points, '
      f'reprojection error {colmap_result.compute_mean_reprojection_error():.2f} px, '
      f'{time.time() - start_time:.0f} s')

if len(colmap_positions) == number_of_photos:
    plot_camera_paths([(positions_ba, 'your SfM + 3 bricks', '#2EAD6B'),
                       (colmap_positions, 'COLMAP', '#4363d8')],
                      'Your SfM vs COLMAP, camera path seen from above')
align_colmap_to_true = make_aligner(colmap_positions, true_positions)
show_3d([(align_colmap_to_true(colmap_cloud), colmap_colors, 'COLMAP')],
        camera_positions=align_colmap_to_true(colmap_positions), title='COLMAP on the fountain')

# **Session 2 · A Geometric Foundation Model: Depth Anything 3**
Same photos, one forward pass of a neural network.
No matching, no PnP, no bundle adjustment. It predicts the cameras and the depth of every pixel.

*The messages `[WARN] gsplat` and `HF_TOKEN` below are harmless.*

In [ ]:
#@title Load Depth Anything 3 (about 1 min)
import warnings, subprocess, sys
warnings.filterwarnings('ignore')   # hides harmless warnings of the libraries
import torch

# Depth Anything 3 needs a few small libraries. If one is missing, install it and try again.
for attempt in range(20):
    try:
        from depth_anything_3.api import DepthAnything3
        break
    except ModuleNotFoundError as missing:
        library = missing.name.split('.')[0]
        print('installing', library)
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '--no-deps', library],
                       capture_output=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('No GPU found: it will work, but slowly. Runtime > Change runtime type > T4 GPU.')
depth_model = DepthAnything3.from_pretrained('depth-anything/DA3-BASE').to(device).eval()
print('Model ready on', device)

In [ ]:
#@title Run Depth Anything 3 on the fountain
MAX_PHOTOS_FOR_DA3 = 20   # keeps the memory of a T4 GPU safe


def run_depth_anything(photo_paths):
    """One forward pass. Returns the camera positions and a colored 3D point cloud."""
    if len(photo_paths) > MAX_PHOTOS_FOR_DA3:
        chosen = np.linspace(0, len(photo_paths) - 1, MAX_PHOTOS_FOR_DA3).astype(int)
        photo_paths = [photo_paths[index] for index in chosen]
    if device == 'cuda':
        torch.cuda.empty_cache()
    with torch.inference_mode():
        prediction = depth_model.inference(photo_paths)

    # Extrinsics map world -> camera. The camera position in the world is -R^T t.
    extrinsics = np.asarray(prediction.extrinsics)[:, :3]
    camera_positions = np.array([-pose[:, :3].T @ pose[:, 3] for pose in extrinsics])

    # Every pixel with a depth becomes a 3D point.
    all_points, all_colors = [], []
    for index in range(len(photo_paths)):
        depth = prediction.depth[index]
        confidence = prediction.conf[index]
        intrinsics = prediction.intrinsics[index]
        rotation, translation = extrinsics[index][:, :3], extrinsics[index][:, 3]

        height, width = depth.shape
        columns, rows = np.meshgrid(np.arange(width), np.arange(height))
        reliable = confidence > np.percentile(confidence, 40)   # drop the 40% least reliable

        pixels = np.stack([columns[reliable], rows[reliable], np.ones(reliable.sum())])
        rays = np.linalg.inv(intrinsics) @ pixels
        points_in_camera = rays * depth[reliable]
        points_in_world = (rotation.T @ (points_in_camera - translation[:, None])).T
        all_points.append(points_in_world)
        all_colors.append(prediction.processed_images[index][reliable])

    run_depth_anything.photo_paths = photo_paths   # remember which photos were used
    run_depth_anything.rotations = extrinsics[:, :, :3]
    return camera_positions, np.vstack(all_points), np.vstack(all_colors).astype(int)


start_time = time.time()
da3_positions, da3_cloud, da3_colors = run_depth_anything(photo_paths)
print(f'Depth Anything 3: {len(photo_paths)} photos in {time.time() - start_time:.1f} s')

paths_to_plot = [(positions_ba, 'your SfM + 3 bricks', '#2EAD6B'),
                 (colmap_positions, 'COLMAP', '#4363d8'),
                 (da3_positions, 'Depth Anything 3', '#f58231')]
plot_camera_paths(paths_to_plot, 'Cameras: your SfM vs COLMAP vs Depth Anything 3')

align_da3_to_true = make_aligner(da3_positions, true_positions)
show_3d([(align_da3_to_true(da3_cloud), da3_colors, 'Depth Anything 3')],
        camera_positions=align_da3_to_true(da3_positions), title='Depth Anything 3 on the fountain')

# **Session 3 · COLMAP vs Depth Anything 3 on our photos**
**Your photos:** 10 to 20 photos of one textured scene. Move around it (do not spin in place).
Zip them and upload the zip (or upload the photos directly). JPEG or PNG.

Nothing uploaded? The fountain is used instead.

We compare 4 numbers for each method, then we try a set of photos that is made to fail.

In [ ]:
#@title 3.1 Upload your photos
from google.colab import files

MAX_PHOTO_SIZE = 1600   # photos are shrunk to this size (in pixels): it keeps COLMAP fast


def load_photos(folder):
    """Asks for an upload, puts the photos in `folder`, returns their paths."""
    shutil.rmtree(folder, ignore_errors=True)
    os.makedirs(folder)
    uploaded = files.upload()
    if len(uploaded) == 0:
        print('Nothing uploaded: using the fountain')
        for path in photo_paths:
            shutil.copy(path, folder)
    for name in uploaded:
        if name.lower().endswith('.zip'):
            os.system(f'unzip -qoj "{name}" -d {folder}')
        else:
            shutil.copy(name, folder)

    # Keep only real photos (zip files from a Mac contain hidden junk files).
    for path in glob.glob(folder + '/*'):
        name = os.path.basename(path)
        if name.startswith('.') or not name.lower().endswith(('.jpg', '.jpeg', '.png')):
            os.remove(path)

    # Shrink big photos.
    for path in glob.glob(folder + '/*'):
        photo = cv2.imread(path)
        if photo is None:
            os.remove(path)
            continue
        biggest_side = max(photo.shape[:2])
        if biggest_side > MAX_PHOTO_SIZE:
            ratio = MAX_PHOTO_SIZE / biggest_side
            photo = cv2.resize(photo, None, fx=ratio, fy=ratio, interpolation=cv2.INTER_AREA)
            cv2.imwrite(path, photo)

    paths = sorted(glob.glob(folder + '/*'), key=photo_number)
    print(len(paths), 'photos')
    return paths


my_photo_paths = load_photos('my_photos')

In [ ]:
#@title 3.2 Compare COLMAP and Depth Anything 3
def compare_methods(paths, name):
    """Runs both methods on the same photos and prints the 4 numbers."""
    photo_folder = os.path.dirname(paths[0])

    # COLMAP
    start_time = time.time()
    try:
        colmap_reconstruction = run_colmap(photo_folder, 'colmap_' + name)
    except Exception as error:
        print('COLMAP failed:', error)
        colmap_reconstruction = None
    colmap_seconds = time.time() - start_time

    # Depth Anything 3
    start_time = time.time()
    da3_positions, da3_cloud, da3_colors = run_depth_anything(paths)
    da3_seconds = time.time() - start_time
    da3_names = [os.path.basename(path) for path in run_depth_anything.photo_paths]

    colmap_placed, colmap_error, colmap_positions_by_name = 0, float('nan'), {}
    if colmap_reconstruction is not None:
        colmap_placed = colmap_reconstruction.num_reg_images()
        colmap_error = colmap_reconstruction.compute_mean_reprojection_error()
        names, positions = colmap_camera_positions(colmap_reconstruction)
        colmap_positions_by_name = dict(zip(names, positions))

    # Do the two methods agree on where the cameras are? Move DA3 cameras onto COLMAP cameras.
    shared = [i for i, photo_name in enumerate(da3_names) if photo_name in colmap_positions_by_name]
    agreement = 'n/a (COLMAP placed fewer than 3 photos)'
    if len(shared) >= 3:
        da3_shared = da3_positions[shared]
        colmap_shared = np.array([colmap_positions_by_name[da3_names[i]] for i in shared])
        da3_moved = make_aligner(da3_shared, colmap_shared)(da3_shared)
        path_size = np.linalg.norm(colmap_shared.max(axis=0) - colmap_shared.min(axis=0))
        gap = np.linalg.norm(da3_moved - colmap_shared, axis=1)
        agreement = f'{100 * np.median(gap) / path_size:.1f} % of the path size'

        plt.figure(figsize=(6, 5))
        plt.plot(colmap_shared[:, 0], colmap_shared[:, 2], '-o', color='#4363d8', label='COLMAP')
        plt.plot(da3_moved[:, 0], da3_moved[:, 2], '-o', color='#f58231', label='Depth Anything 3')
        plt.gca().set_aspect('equal', 'datalim')
        plt.legend()
        plt.title(f'Cameras seen from above: {name}')
        plt.show()

    if colmap_reconstruction is not None and colmap_reconstruction.num_points3D() > 0:
        colmap_cloud, colmap_colors = colmap_points(colmap_reconstruction)
        show_3d([(colmap_cloud, colmap_colors, 'COLMAP')], camera_positions=positions,
                camera_rotations=colmap_camera_rotations(colmap_reconstruction),
                title=f'COLMAP: {name}')
    show_3d([(da3_cloud, da3_colors, 'Depth Anything 3')], camera_positions=da3_positions,
            camera_rotations=run_depth_anything.rotations, title=f'Depth Anything 3: {name}')

    table = pd.DataFrame({
        'COLMAP': [f'{colmap_placed} / {len(paths)}', f'{colmap_error:.2f} px',
                   'reference', f'{colmap_seconds:.0f} s'],
        'Depth Anything 3': [f'{len(da3_names)} / {len(paths)}', 'no such measure (no matching)',
                             agreement, f'{da3_seconds:.1f} s']},
        index=['1 · Photos placed', '2 · Reprojection error (lower is better)',
               '3 · Cameras: agreement with COLMAP', '4 · Time'])
    display(table)
    return table


my_table = compare_methods(my_photo_paths, 'our_photos')

**What to write down:** the 4 numbers for each method.
Which method placed all the photos? Which one was faster? Do the two 3D views look alike?

## **3.3 · Trap round**
Upload a **bad** set of photos: spinning in place without moving, a white wall, or only 3 photos.
Which method breaks first?

In [ ]:
#@title 3.3 Upload the trap set and compare
trap_photo_paths = load_photos('trap_photos')
trap_table = compare_methods(trap_photo_paths, 'trap_set')